In [1]:
from typing import Dict, Sequence

import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.01
TABLE_METRICS = [
    "accuracy_seen_avg",
    "accuracy_final",
    "backward_transfer",
    "ece_seen_avg",
    "ece_final",
    # "sce_final",
    "asce_final",
    # "brier_final",
]
FMT_STRS = {
    "_": ("{:02.2f}", "{:.2f}"),
    "sce_final": ("{:.3f}", "{:.3f}"),
    "asce_final": ("{:.3f}", "{:.3f}"),
}

LOWER_IS_BETTER = {
    "ece_seen_avg",
    "ece_all_avg",
    "ece_final",
    "ace_final",
    "sce_final",
    "brier_final",
    "asce_final",
}


df = pd.read_parquet("dataframe/summary.parquet")
df.head(4)

,dataset,method,run_id,n_tasks,accuracy_all_avg,accuracy_final,accuracy_seen_avg,ace_all_avg,ace_final,ace_seen_avg,...,brier_all_avg,brier_final,ece_final,ece_seen_avg,ece_all_avg,forward_transfer,sce_all_avg,sce_final,sce_seen_avg,asce_final
0,cifar100,linear,2917091_1,10,0.40325,0.6643,0.778219,0.008889,0.000091,0.000036,...,0.865032,0.450561,0.032468,0.023133,0.280823,0.0,0.007849,0.000609,0.000401,0.000608
1,cifar100,linear,2917080_4,10,0.40310,0.6685,0.777365,0.008888,0.000039,0.000028,...,0.861300,0.441696,0.028633,0.020610,0.279394,0.0,0.007769,0.000505,0.000341,0.000494
2,cifar100,linear,2917092_2,10,0.40042,0.6621,0.773784,0.008874,0.000052,0.000028,...,0.864492,0.448588,0.035262,0.024930,0.283331,0.0,0.007848,0.000662,0.000424,0.000655
3,cifar100,linear,2916795_0,10,0.40130,0.6703,0.772594,0.008899,0.000051,0.000021,...,0.866215,0.442307,0.024716,0.021827,0.282382,0.0,0.007816,0.000476,0.000379,0.000477


In [2]:
def significance_test(
    df: pd.DataFrame, metric: str, reference: str
) -> Dict[str, tuple[float, float]]:
    equal_var = False
    samples: Sequence[np.ndarray] = []
    methods: Sequence[str] = []

    for method, group in df.groupby("method"):
        assert isinstance(method, str)
        group_samples = np.abs(group[metric].to_numpy())
        if np.any(np.isnan(group_samples)):
            continue
        samples.append(group_samples)
        methods.append(method)

    # One way ANOVA
    f_oneway_result = stats.f_oneway(*samples, equal_var=equal_var)
    if f_oneway_result.pvalue >= ALPHA:
        print(
            f"WARNING: ANOVA test for metric {metric} is not significant (p={f_oneway_result.pvalue:.4f})"
        )
    else:
        print(
            f"ANOVA test for metric {metric} is significant (p={f_oneway_result.pvalue:.4f})"
        )

    reference_idx = methods.index(reference)
    result = stats.tukey_hsd(*samples, equal_var=equal_var)

    return {
        method: (result.pvalue[i, reference_idx], result.statistic[i, reference_idx])
        for i, method in enumerate(methods)
    }

In [3]:
from common import METHOD_TO_LABEL, METRIC_TO_LABEL, OFFLINE, OFFLINE_COMPATIBLE_METRICS


def to_display_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    p_values: Dict[str, Dict[str, tuple[float, float]]] = {}
    for m in TABLE_METRICS:
        p_values[m] = significance_test(df, m, reference="ball")

    df = df.groupby("method").aggregate({m: ["mean", "std"] for m in TABLE_METRICS})

    fair_mask = df.index.to_series().apply(lambda x: x not in OFFLINE)

    rows = []

    for method, data in df.iterrows():
        method: str

        row = {
            "method": METHOD_TO_LABEL[method],
        }
        for key in TABLE_METRICS:
            p_value, test_statistic = p_values[key].get(method, (1.0, 0.0))
            label = METRIC_TO_LABEL.get(key, key)
            col_max = df[fair_mask][key]["mean"].max()
            col_min = df[fair_mask][key]["mean"].min()

            value = data[(key, "mean")]
            mean = float(value * 100)
            std = data[(key, "std")] * 100

            # if std is NaN, replace with 0.0
            if pd.isna(std):
                std = 0.0

            # Determine if this value should be bolded
            if key not in LOWER_IS_BETTER:
                bold = value >= col_max
            else:
                bold = value <= col_min

            # If the method is in UNFAIR_COMPARISONS, never bold
            italics = False
            if method in OFFLINE:
                bold = False
                italics = True

            fmt_mean, fmt_std = FMT_STRS.get(key) or FMT_STRS["_"]
            label = rf"{{\small {label}}}"
            row[label] = rf"{fmt_mean.format(mean)} $\pm$ {fmt_std.format(std)}"
            if bold:
                row[label] = rf"\textbf{{{row[label]}}}"
            if italics:
                row[label] = rf"\textit{{{row[label]}}}"

            # Significance annotation
            sig_higher = r"$^{\uparrow}$"
            sig_lower = r"$^{\downarrow}$"
            if p_value < ALPHA:
                # The test-statistic points the wrong way when the mean is negative
                # Maybe an absolute value is taken somewhere?
                if mean < 0:
                    test_statistic = -test_statistic
                if test_statistic > 0:
                    row[label] += sig_higher
                else:
                    row[label] += sig_lower

            row[label] = r"\small " + row[label]

            if method in OFFLINE and key not in OFFLINE_COMPATIBLE_METRICS:
                row[label] = r"{\small \textit{NA}}"

        rows.append(row)

    table = pd.DataFrame(rows)
    table = table.set_index("method")
    # make table follow a specific order
    table = table.reindex(list(METHOD_TO_LABEL.values()))

    # table.columns.names = [None, None]
    table.index.name = None
    return table

In [4]:
for dataset in df["dataset"].unique():
    dataset_table = to_display_table(df[df["dataset"] == dataset])
    filename = f"tables/{dataset}.tex"
    with open(filename, "w") as f:
        dataset_table.to_latex(
            f, column_format="l" + "l" * (len(dataset_table.columns))
        )
    print(f"Saved table for {dataset} to {filename}")

ANOVA test for metric accuracy_seen_avg is significant (p=0.0000)
ANOVA test for metric accuracy_final is significant (p=0.0000)
ANOVA test for metric backward_transfer is significant (p=0.0000)
ANOVA test for metric ece_seen_avg is significant (p=0.0000)
ANOVA test for metric ece_final is significant (p=0.0000)
ANOVA test for metric asce_final is significant (p=0.0000)
Saved table for cifar100 to tables/cifar100.tex
ANOVA test for metric accuracy_seen_avg is significant (p=0.0000)
ANOVA test for metric accuracy_final is significant (p=0.0000)
ANOVA test for metric backward_transfer is significant (p=0.0000)
ANOVA test for metric ece_seen_avg is significant (p=0.0000)
ANOVA test for metric ece_final is significant (p=0.0000)
ANOVA test for metric asce_final is significant (p=0.0000)
Saved table for domainnet to tables/domainnet.tex
ANOVA test for metric accuracy_seen_avg is significant (p=0.0000)
ANOVA test for metric accuracy_final is significant (p=0.0000)
ANOVA test for metric backw